# 工具解析器

本指南演示如何使用 SGLang 的[函数调用](https://platform.openai.com/docs/guides/function-calling)功能。

## 当前支持的解析器：

| 解析器 | 支持的模型 | 备注 |
|---|---|---|
| `deepseekv3` | DeepSeek-v3（例如 `deepseek-ai/DeepSeek-V3-0324`） | 建议在启动命令中添加 `--chat-template ./examples/chat_template/tool_chat_template_deepseekv3.jinja`。 |
| `deepseekv31` | DeepSeek-V3.1 和 DeepSeek-V3.2-Exp（例如 `deepseek-ai/DeepSeek-V3.1`、`deepseek-ai/DeepSeek-V3.2-Exp`） | 建议在启动命令中添加 `--chat-template ./examples/chat_template/tool_chat_template_deepseekv31.jinja`（或对 DeepSeek-V3.2 使用 ..deepseekv32.jinja）。 |
| `deepseekv32` | DeepSeek-V3.2（`deepseek-ai/DeepSeek-V3.2`） | |
| `glm` | GLM 系列（例如 `zai-org/GLM-4.6`） | |
| `gpt-oss` | GPT-OSS（例如 `openai/gpt-oss-120b`、`openai/gpt-oss-20b`、`lmsys/gpt-oss-120b-bf16`、`lmsys/gpt-oss-20b-bf16`） | gpt-oss 工具解析器会过滤分析通道事件，仅保留正常文本。当解释在分析通道中时，这可能导致内容为空。解决方法是通过返回 `role="tool"` 消息作为工具结果来完成工具轮次，使模型生成最终内容。 |
| `kimi_k2` | `moonshotai/Kimi-K2-Instruct` | |
| `llama3` | Llama 3.1 / 3.2 / 3.3（例如 `meta-llama/Llama-3.1-8B-Instruct`、`meta-llama/Llama-3.2-1B-Instruct`、`meta-llama/Llama-3.3-70B-Instruct`） | |
| `llama4` | Llama 4（例如 `meta-llama/Llama-4-Scout-17B-16E-Instruct`） | |
| `mistral` | Mistral（例如 `mistralai/Mistral-7B-Instruct-v0.3`、`mistralai/Mistral-Nemo-Instruct-2407`、`mistralai/Mistral-7B-v0.3`） | |
| `pythonic` | Llama-3.2 / Llama-3.3 / Llama-4 | 模型以 Python 代码形式输出函数调用。需要 `--tool-call-parser pythonic`，建议配合特定的聊天模板使用。 |
| `qwen` | Qwen 系列（例如 `Qwen/Qwen3-Next-80B-A3B-Instruct`、`Qwen/Qwen3-VL-30B-A3B-Thinking`），Qwen3-Coder 除外 | |
| `qwen3_coder` | Qwen3-Coder（例如 `Qwen/Qwen3-Coder-30B-A3B-Instruct`） | |
| `step3` | Step-3 | |


## OpenAI 兼容 API

### 启动服务器

In [ ]:
import json
from sglang.test.doc_patch import launch_server_cmd
from sglang.utils import wait_for_server, print_highlight, terminate_process
from openai import OpenAI

server_process, port = launch_server_cmd(
    "python3 -m sglang.launch_server --model-path Qwen/Qwen2.5-7B-Instruct --tool-call-parser qwen25 --host 0.0.0.0 --log-level warning"  # qwen25
)
wait_for_server(f"http://localhost:{port}", process=server_process)

请注意，`--tool-call-parser` 定义了用于解释响应的解析器。

### 定义函数调用的工具
以下是一个 Python 代码片段，展示如何将工具定义为字典。字典包含工具名称、描述和属性定义的参数。

In [ ]:
# Define tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city to find the weather for, e.g. 'San Francisco'",
                    },
                    "state": {
                        "type": "string",
                        "description": "the two-letter abbreviation for the state that the city is"
                        " in, e.g. 'CA' which would mean 'California'",
                    },
                    "unit": {
                        "type": "string",
                        "description": "The unit to fetch the temperature in",
                        "enum": ["celsius", "fahrenheit"],
                    },
                },
                "required": ["city", "state", "unit"],
            },
        },
    }
]

### 定义消息

In [ ]:
def get_messages():
    return [
        {
            "role": "user",
            "content": "What's the weather like in Boston today? Output a reasoning before act, then use the tools to help you.",
        }
    ]


messages = get_messages()

### 初始化客户端

In [ ]:
# Initialize OpenAI-like client
client = OpenAI(api_key="None", base_url=f"http://0.0.0.0:{port}/v1")
model_name = client.models.list().data[0].id

### 非流式请求

In [ ]:
# Non-streaming mode test
response_non_stream = client.chat.completions.create(
    model=model_name,
    messages=messages,
    temperature=0,
    top_p=0.95,
    max_tokens=1024,
    stream=False,  # Non-streaming
    tools=tools,
)
print_highlight("Non-stream response:")
print_highlight(response_non_stream)
print_highlight("==== content ====")
print_highlight(response_non_stream.choices[0].message.content)
print_highlight("==== tool_calls ====")
print_highlight(response_non_stream.choices[0].message.tool_calls)

#### 处理工具
当引擎决定应调用特定工具时，它会通过响应返回参数或部分参数。您可以解析这些参数，然后相应地调用工具。

In [ ]:
name_non_stream = response_non_stream.choices[0].message.tool_calls[0].function.name
arguments_non_stream = (
    response_non_stream.choices[0].message.tool_calls[0].function.arguments
)

print_highlight(f"Final streamed function call name: {name_non_stream}")
print_highlight(f"Final streamed function call arguments: {arguments_non_stream}")

### 流式请求

In [ ]:
# Streaming mode test
print_highlight("Streaming response:")
response_stream = client.chat.completions.create(
    model=model_name,
    messages=messages,
    temperature=0,
    top_p=0.95,
    max_tokens=1024,
    stream=True,  # Enable streaming
    tools=tools,
)

texts = ""
tool_calls = []
name = ""
arguments = ""
for chunk in response_stream:
    if chunk.choices[0].delta.content:
        texts += chunk.choices[0].delta.content
    if chunk.choices[0].delta.tool_calls:
        tool_calls.append(chunk.choices[0].delta.tool_calls[0])
print_highlight("==== Text ====")
print_highlight(texts)

print_highlight("==== Tool Call ====")
for tool_call in tool_calls:
    print_highlight(tool_call)

#### 处理工具
当引擎决定应调用特定工具时，它会通过响应返回参数或部分参数。您可以解析这些参数，然后相应地调用工具。

In [ ]:
# Parse and combine function call arguments
arguments = []
for tool_call in tool_calls:
    if tool_call.function.name:
        print_highlight(f"Streamed function call name: {tool_call.function.name}")

    if tool_call.function.arguments:
        arguments.append(tool_call.function.arguments)

# Combine all fragments into a single JSON string
full_arguments = "".join(arguments)
print_highlight(f"streamed function call arguments: {full_arguments}")

### 定义工具函数

In [ ]:
# This is a demonstration, define real function according to your usage.
def get_current_weather(city: str, state: str, unit: "str"):
    return (
        f"The weather in {city}, {state} is 85 degrees {unit}. It is "
        "partly cloudly, with highs in the 90's."
    )


available_tools = {"get_current_weather": get_current_weather}


### 执行工具

In [ ]:
messages.append(response_non_stream.choices[0].message)

# Call the corresponding tool function
tool_call = messages[-1].tool_calls[0]
tool_name = tool_call.function.name
tool_to_call = available_tools[tool_name]
result = tool_to_call(**(json.loads(tool_call.function.arguments)))
print_highlight(f"Function call result: {result}")
# messages.append({"role": "tool", "content": result, "name": tool_name})
messages.append(
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(result),
        "name": tool_name,
    }
)

print_highlight(f"Updated message history: {messages}")

### 将结果发送回模型

In [ ]:
final_response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    temperature=0,
    top_p=0.95,
    stream=False,
    tools=tools,
)
print_highlight("Non-stream response:")
print_highlight(final_response)

print_highlight("==== Text ====")
print_highlight(final_response.choices[0].message.content)

## 原生 API 和 SGLang 运行时（SRT）

In [ ]:
from transformers import AutoTokenizer
import requests

# generate an answer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

messages = get_messages()

input = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, tools=tools, return_dict=False
)

gen_url = f"http://localhost:{port}/generate"
gen_data = {
    "text": input,
    "sampling_params": {
        "skip_special_tokens": False,
        "max_new_tokens": 1024,
        "temperature": 0,
        "top_p": 0.95,
    },
}
gen_response = requests.post(gen_url, json=gen_data).json()["text"]
print_highlight("==== Response ====")
print_highlight(gen_response)

# parse the response
parse_url = f"http://localhost:{port}/parse_function_call"

function_call_input = {
    "text": gen_response,
    "tool_call_parser": "qwen25",
    "tools": tools,
}

function_call_response = requests.post(parse_url, json=function_call_input)
function_call_response_json = function_call_response.json()

print_highlight("==== Text ====")
print(function_call_response_json["normal_text"])
print_highlight("==== Calls ====")
print("function name: ", function_call_response_json["calls"][0]["name"])
print("function arguments: ", function_call_response_json["calls"][0]["parameters"])

In [ ]:
terminate_process(server_process)

## 离线引擎 API

In [ ]:
import sglang as sgl
from sglang.srt.function_call.function_call_parser import FunctionCallParser
from sglang.srt.managers.io_struct import Tool, Function

llm = sgl.Engine(model_path="Qwen/Qwen2.5-7B-Instruct")
tokenizer = llm.tokenizer_manager.tokenizer
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, tools=tools, return_dict=False
)

# Note that for gpt-oss tool parser, adding "no_stop_trim": True
# to make sure the tool call token <call> is not trimmed.

sampling_params = {
    "max_new_tokens": 1024,
    "temperature": 0,
    "top_p": 0.95,
    "skip_special_tokens": False,
}

# 1) Offline generation
result = llm.generate(input_ids=input_ids, sampling_params=sampling_params)
generated_text = result["text"]  # Assume there is only one prompt

print_highlight("=== Offline Engine Output Text ===")
print_highlight(generated_text)


# 2) Parse using FunctionCallParser
def convert_dict_to_tool(tool_dict: dict) -> Tool:
    function_dict = tool_dict.get("function", {})
    return Tool(
        type=tool_dict.get("type", "function"),
        function=Function(
            name=function_dict.get("name"),
            description=function_dict.get("description"),
            parameters=function_dict.get("parameters"),
        ),
    )


tools = [convert_dict_to_tool(raw_tool) for raw_tool in tools]

parser = FunctionCallParser(tools=tools, tool_call_parser="qwen25")
normal_text, calls = parser.parse_non_stream(generated_text)

print_highlight("=== Parsing Result ===")
print("Normal text portion:", normal_text)
print_highlight("Function call portion:")
for call in calls:
    # call: ToolCallItem
    print_highlight(f"  - tool name: {call.name}")
    print_highlight(f"    parameters: {call.parameters}")

# 3) If needed, perform additional logic on the parsed functions, such as automatically calling the corresponding function to obtain a return value, etc.

In [ ]:
llm.shutdown()

## 工具选择模式

SGLang 支持 OpenAI 的 `tool_choice` 参数来控制模型何时以及调用哪些工具。此功能使用 EBNF（扩展巴科斯-瑙尔范式）语法实现，以确保可靠的工具调用行为。

### 支持的工具选择选项

- **`tool_choice="required"`**：强制模型至少调用一个工具
- **`tool_choice={"type": "function", "function": {"name": "specific_function"}}`**：强制模型调用特定函数

### 后端兼容性

工具选择在 **Xgrammar 后端**（默认语法后端 `--grammar-backend xgrammar`）上完全支持。但在其他后端（如 `outlines`）上可能不完全支持。

### 示例：必需工具选择

In [ ]:
from openai import OpenAI
from sglang.utils import wait_for_server, print_highlight, terminate_process
from sglang.test.doc_patch import launch_server_cmd

# Start a new server session for tool choice examples
server_process_tool_choice, port_tool_choice = launch_server_cmd(
    "python3 -m sglang.launch_server --model-path Qwen/Qwen2.5-7B-Instruct --tool-call-parser qwen25 --host 0.0.0.0  --log-level warning"
)
wait_for_server(
    f"http://localhost:{port_tool_choice}", process=server_process_tool_choice
)

# Initialize client for tool choice examples
client_tool_choice = OpenAI(
    api_key="None", base_url=f"http://0.0.0.0:{port_tool_choice}/v1"
)
model_name_tool_choice = client_tool_choice.models.list().data[0].id

# Example with tool_choice="required" - forces the model to call a tool
messages_required = [
    {"role": "user", "content": "Hello, what is the capital of France?"}
]

# Define tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city to find the weather for, e.g. 'San Francisco'",
                    },
                    "unit": {
                        "type": "string",
                        "description": "The unit to fetch the temperature in",
                        "enum": ["celsius", "fahrenheit"],
                    },
                },
                "required": ["city", "unit"],
            },
        },
    }
]

response_required = client_tool_choice.chat.completions.create(
    model=model_name_tool_choice,
    messages=messages_required,
    temperature=0,
    max_tokens=1024,
    tools=tools,
    tool_choice="required",  # Force the model to call a tool
)

print_highlight("Response with tool_choice='required':")
print("Content:", response_required.choices[0].message.content)
print("Tool calls:", response_required.choices[0].message.tool_calls)

### 示例：指定函数选择


In [ ]:
# Example with specific function choice - forces the model to call a specific function
messages_specific = [
    {"role": "user", "content": "What are the most attactive places in France?"}
]

response_specific = client_tool_choice.chat.completions.create(
    model=model_name_tool_choice,
    messages=messages_specific,
    temperature=0,
    max_tokens=1024,
    tools=tools,
    tool_choice={
        "type": "function",
        "function": {"name": "get_current_weather"},
    },  # Force the model to call the specific get_current_weather function
)

print_highlight("Response with specific function choice:")
print("Content:", response_specific.choices[0].message.content)
print("Tool calls:", response_specific.choices[0].message.tool_calls)

if response_specific.choices[0].message.tool_calls:
    tool_call = response_specific.choices[0].message.tool_calls[0]
    print_highlight(f"Called function: {tool_call.function.name}")
    print_highlight(f"Arguments: {tool_call.function.arguments}")

In [ ]:
terminate_process(server_process_tool_choice)

## Pythonic 工具调用格式（Llama-3.2 / Llama-3.3 / Llama-4）

某些 Llama 模型（如 Llama-3.2-1B、Llama-3.2-3B、Llama-3.3-70B 和 Llama-4）支持"pythonic"工具调用格式，其中模型以 Python 代码形式输出函数调用，例如：

```python
[get_current_weather(city="San Francisco", state="CA", unit="celsius")]
```

- 输出是一个 Python 函数调用列表，参数为 Python 字面量（不是 JSON）。
- 同一列表中可以返回多个工具调用：
```python
[get_current_weather(city="San Francisco", state="CA", unit="celsius"),
 get_current_weather(city="New York", state="NY", unit="fahrenheit")]
```

更多信息，请参阅 Meta 关于[零样本函数调用](https://github.com/meta-llama/llama-models/blob/main/models/llama4/prompt_format.md#zero-shot-function-calling---system-message)的文档。

请注意，此功能在 Blackwell 上仍在开发中。

### 如何启用
- 使用 `--tool-call-parser pythonic` 启动服务器
- 您也可以使用 --chat-template 指定改进的模型模板（例如 `--chat-template=examples/chat_template/tool_chat_template_llama4_pythonic.jinja`）。
建议这样做，因为模型期望特定的提示格式才能可靠地生成有效的 pythonic 工具调用输出。模板确保提示结构（例如特殊标记、消息边界如 `<|eom|>` 和函数调用分隔符）与模型训练或微调时使用的格式匹配。如果不使用正确的聊天模板，工具调用可能失败或产生不一致的结果。

#### 不使用聊天模板强制 Pythonic 工具调用输出
如果您不想指定聊天模板，则必须在消息中给模型极其明确的指令来强制 pythonic 输出。例如，对于 `Llama-3.2-1B-Instruct`，您需要：

In [ ]:
import openai

server_process, port = launch_server_cmd(
    " python3 -m sglang.launch_server --model-path meta-llama/Llama-3.2-1B-Instruct --tool-call-parser pythonic --tp 1  --log-level warning"  # llama-3.2-1b-instruct
)
wait_for_server(f"http://localhost:{port}", process=server_process)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The name of the city or location.",
                    }
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_tourist_attractions",
            "description": "Get a list of top tourist attractions for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to find attractions for.",
                    }
                },
                "required": ["city"],
            },
        },
    },
]


def get_messages():
    return [
        {
            "role": "system",
            "content": (
                "You are a travel assistant. "
                "When asked to call functions, ALWAYS respond ONLY with a python list of function calls, "
                "using this format: [func_name1(param1=value1, param2=value2), func_name2(param=value)]. "
                "Do NOT use JSON, do NOT use variables, do NOT use any other format. "
                "Here is an example:\n"
                '[get_weather(location="Paris"), get_tourist_attractions(city="Paris")]'
            ),
        },
        {
            "role": "user",
            "content": (
                "I'm planning a trip to Tokyo next week. What's the weather like and what are some top tourist attractions? "
                "Propose parallel tool calls at once, using the python list of function calls format as shown above."
            ),
        },
    ]


messages = get_messages()

client = openai.Client(base_url=f"http://localhost:{port}/v1", api_key="xxxxxx")
model_name = client.models.list().data[0].id


response_non_stream = client.chat.completions.create(
    model=model_name,
    messages=messages,
    temperature=0,
    top_p=0.9,
    stream=False,  # Non-streaming
    tools=tools,
)
print_highlight("Non-stream response:")
print_highlight(response_non_stream)

response_stream = client.chat.completions.create(
    model=model_name,
    messages=messages,
    temperature=0,
    top_p=0.9,
    stream=True,
    tools=tools,
)
texts = ""
tool_calls = []
name = ""
arguments = ""

for chunk in response_stream:
    if chunk.choices[0].delta.content:
        texts += chunk.choices[0].delta.content
    if chunk.choices[0].delta.tool_calls:
        tool_calls.append(chunk.choices[0].delta.tool_calls[0])

print_highlight("Streaming Response:")
print_highlight("==== Text ====")
print_highlight(texts)

print_highlight("==== Tool Call ====")
for tool_call in tool_calls:
    print_highlight(tool_call)

terminate_process(server_process)

> **注意：**
> 如果模型在 JSON 格式上进行了大量微调，它可能仍会默认使用 JSON。如果不使用聊天模板，提示工程（包括示例）是增加 pythonic 输出概率的唯一方法。

## 如何支持新模型？
1. 在 sglang/srt/function_call_parser.py 中更新 TOOLS_TAG_LIST，添加模型的工具标签。目前支持的标签包括：
```
	TOOLS_TAG_LIST = [
	    "<|plugin|>",
	    "<function=",
	    "<tool_call>",
	    "<|python_tag|>",
	    "[TOOL_CALLS]"
	]
```
2. 在 sglang/srt/function_call_parser.py 中创建一个新的检测器类，继承自 BaseFormatDetector。检测器应处理模型特定的函数调用格式。例如：
```
    class NewModelDetector(BaseFormatDetector):
```
3. 将新检测器添加到管理所有格式检测器的 MultiFormatParser 类中。